# Titanic Data Workflow

**Name:** Christian

**Dataset:** Titanic (seaborn-data copy, titanic.csv)

This project builds a reproducible data workflow in Python: loading, cleaning, exploring and visualizing the Titanic passenger dataset. The goal is to understand which passenger characteristics were associated with survival. It also practices modular code, documentation and Git version control.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Data Ingestion

In [2]:
df = pd.read_csv("titanic.csv")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 3. Data Cleaning

Before cleaning, I check how much data is missing in each column.

In [3]:
print(df.isna().sum())
print('Rows, columns:', df.shape)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
Rows, columns: (891, 15)


In [4]:
def standardize_columns(data):
    """Standardize text values and remove a redundant column.

    Strips whitespace and lowercases every text column so category labels
    are consistent, and drops 'alive', which duplicates the 'survived'
    column. Returns a new DataFrame; the input is not modified.
    """
    data = data.copy()
    data.columns = data.columns.str.strip().str.lower()
    for col in data.columns:
        if pd.api.types.is_string_dtype(data[col]) or data[col].dtype == object:
            data[col] = data[col].str.strip().str.lower()
    return data.drop(columns=["alive"], errors="ignore")


def handle_missing_values(data):
    """Handle missing values in the Titanic dataset.

    - age: filled with the median age of passengers who share the same
      sex and passenger class (more realistic than one global median).
    - embarked / embark_town: the 2 missing rows are filled with the mode.
    - deck: dropped, because about 77% of its values are missing.
    Returns a new DataFrame; the input is not modified.
    """
    data = data.copy()
    data["age"] = data["age"].fillna(
        data.groupby(["sex", "pclass"])["age"].transform("median")
    )
    for col in ["embarked", "embark_town"]:
        data[col] = data[col].fillna(data[col].mode()[0])
    return data.drop(columns=["deck"], errors="ignore")


df_clean = handle_missing_values(standardize_columns(df))
print(df_clean.isna().sum())
print('Rows, columns:', df_clean.shape)
df_clean.head()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alone          0
dtype: int64
Rows, columns: (891, 13)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,s,third,man,True,southampton,False
1,1,1,female,38.0,1,0,71.2833,c,first,woman,False,cherbourg,False
2,1,3,female,26.0,0,0,7.9250,s,third,woman,False,southampton,True
3,1,1,female,35.0,1,0,53.1000,s,first,woman,False,southampton,False
4,0,3,male,35.0,0,0,8.0500,s,third,man,True,southampton,True


### Cleaning Justification

- **Why clean at all:** Missing values break many calculations and can silently distort averages and plots. The `age` column had 177 missing values (about 20%), `embarked` and `embark_town` had 2 each, and `deck` was missing for about 77% of passengers.
- **Age:** I filled missing ages with the median for the passenger's sex and class instead of dropping rows, so I keep all 891 passengers and avoid losing a large share of the data. The median is used because it is less affected by outliers than the mean.
- **Embarked:** Only 2 values were missing, so filling with the most common port is a very small assumption.
- **Deck:** With about 77% missing, any fill would be mostly invented, so I dropped the column.
- **Standardizing text:** Lowercasing and stripping whitespace prevents the same category from being counted as two different labels. I dropped `alive` because it duplicates `survived`.
- **Not removed:** I kept extreme `fare` values because they are real first-class tickets, not errors.